# V40 DATA EXCHANGE — FULL GPU ARM (B1 crucible)
Bar: TS-val CE <= **2.2706** (v38e uniform-allocation control @ 13ep).

**Runtime must be T4 GPU** (preselected below — if it connects as CPU, Runtime > Change runtime type > T4).

Run Cell 1, then Cell 2, keep the tab open (~3h). Verdict prints at the end and lands in `/content/v40_report.json`.

In [ ]:
# ==== V40 CELL 1 — setup (T4 GPU; ~3-5 min total) ====
import os, json, sys, time, subprocess
T0 = time.time()
def ts(): return f"[t+{int(time.time()-T0):>3}s]"

# ---- 1) creds: pick the file in the widget (or CANCEL -> paste key). self-healing ----
tok = None
if os.path.exists("/root/.kaggle/kaggle.json"):
    try:
        tok = json.load(open("/root/.kaggle/kaggle.json"))
        if not (tok.get("username") and tok.get("key")): tok = None
    except Exception:
        tok = None
if tok is None:
    print(ts(), "[creds] pick kaggle.json in the widget (or CANCEL -> paste key)")
    try:
        from google.colab import files
        up = files.upload()
        if up:
            k = next(iter(up)); tok = json.loads(up[k].decode())
    except Exception as e:
        print(ts(), "[creds] upload failed:", str(e)[:120])
    if tok is None:
        import getpass
        tok = {"username": "albanchigozirim",
               "key": getpass.getpass("paste ONLY the key string (no quotes): ").strip().strip('"')}
    assert tok.get("username") and tok.get("key"), "invalid token"
    open("/root/.kaggle/kaggle.json", "w").write(json.dumps(tok))
os.chmod("/root/.kaggle/kaggle.json", 0o600)
os.environ["KAGGLE_USERNAME"] = tok["username"]; os.environ["KAGGLE_KEY"] = tok["key"]
print(ts(), f"[creds] ok user={tok['username']} key=...{tok['key'][-4:]}")

# ---- 2) fetch 5 datasets in PARALLEL (~350MB total, ~1-3 min; progress prints as each lands) ----
print(ts(), "[fetch] downloading 5 datasets in parallel (~350MB) — expect ~1-3 min ...", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
import kagglehub, shutil
from concurrent.futures import ThreadPoolExecutor
os.makedirs("/content/data", exist_ok=True)
SLUGS = ["tinystories-gate2-data", "tinystories-gate3-data", "ag-news-v1",
         "curriculum-v37-ckpt", "v40-exchange-src"]
def fetch(slug):
    t = time.time()
    p = kagglehub.dataset_download(f"albanchigozirim/{slug}")
    shutil.copytree(p, "/content/data", dirs_exist_ok=True)
    return f"{ts()} [fetch] {slug} ok ({time.time()-t:.0f}s)"
with ThreadPoolExecutor(5) as ex:
    for msg in ex.map(fetch, SLUGS):
        print(msg, flush=True)
os.environ["V40_DATA"] = "/content/data"

# ---- 3) ANCHOR + ASSERT: exchange plumbing verified before anything runs ----
KERNEL = "/content/data/kernel.py"
assert os.path.exists(KERNEL), "kernel.py missing from v40-exchange-src"
src = open(KERNEL).read()
for tag in ['alloc_ledger.append', 'price_hist.append', 'FLOOR)', 'KAPPA_P',
            'CLEAR=10 if V40_SMOKE else 100', 'np.add.at(contrib', 'mkt_shares=ns']:
    assert tag in src, f"EXCHANGE PLUMBING MISSING: {tag}"
src = src.replace('os.environ.get("V40_SMOKE","1")=="1"', '"0"=="1"')
assert '"0"=="1"' in src, "smoke->full switch failed"
open(KERNEL, "w").write(src)
print(ts(), "[assert] plumbing verified; FULL ARM armed")

# ---- 4) wheel probe: real Mamba required (control 2.2706 ran on real Mamba) ----
import torch
print(ts(), "torch", torch.__version__, "|", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
try:
    from mamba_ssm import Mamba; print(ts(), "[probe] mamba-ssm importable")
except Exception:
    print(ts(), "[probe] installing Mamba wheels (~100MB) ...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--no-deps",
      "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
      "https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"],
      capture_output=True, text=True)
    try:
        from mamba_ssm import Mamba; print(ts(), "[probe] wheels OK")
    except Exception as e:
        print(ts(), "[probe] WHEEL MISMATCH — run: !pip install causal-conv1d==1.7.0 mamba-ssm==2.3.2.post1 --no-build-isolation")
        print("   detail:", str(e)[:200])

# ---- 5) TELEMETRY BEACON (optional — arms only if creds work; never blocks the run) ----
import shutil, threading
_TEL = {"ds": "albanchigozirim/v40-colab-telemetry", "tmp": "/tmp/beacon_push"}
def _beacon_loop():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], capture_output=True)
    while True:
        time.sleep(600)
        try:
            os.makedirs(_TEL["tmp"] + "/v40", exist_ok=True)
            for s, d in [("/content/v40_run.log", "run.log"), ("/content/v40_report.json", "report.json")]:
                if os.path.exists(s): shutil.copy(s, _TEL["tmp"] + "/v40/" + d)
            open(_TEL["tmp"] + "/v40/beat.txt", "w").write(time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()))
            json.dump({"title": "v40-colab-telemetry", "id": _TEL["ds"], "licenses": [{"name": "CC0-1.0"}]},
                      open(_TEL["tmp"] + "/dataset-metadata.json", "w"))
            subprocess.run([sys.executable, "-m", "kaggle", "datasets", "version", "-p", _TEL["tmp"], "-m", "beat"],
                           capture_output=True)
        except Exception:
            pass
threading.Thread(target=_beacon_loop, daemon=True).start()
print(ts(), f"[beacon] armed — setup DONE in {time.time()-T0:.0f}s. Now run CELL 2 (~3h).")


In [ ]:
import subprocess, sys, os, json, shutil, time
p = subprocess.run([sys.executable, "/content/data/kernel.py"], stderr=subprocess.STDOUT)
print(f"\nKERNEL EXIT CODE: {p.returncode}", flush=True)
def push_telemetry():
    try:
        os.makedirs("/tmp/beacon_push/v40", exist_ok=True)
        for s, d in [("/content/v40_run.log", "run.log"), ("/content/v40_report.json", "report.json")]:
            if os.path.exists(s): shutil.copy(s, "/tmp/beacon_push/v40/" + d)
        open("/tmp/beacon_push/v40/beat.txt", "w").write(time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()))
        json.dump({"title": "v40-colab-telemetry", "id": "albanchigozirim/v40-colab-telemetry",
                   "licenses": [{"name": "CC0-1.0"}]},
                  open("/tmp/beacon_push/dataset-metadata.json", "w"))
        q = subprocess.run([sys.executable, "-m", "kaggle", "datasets", "version",
                            "-p", "/tmp/beacon_push", "-m", "cell2-autopush"],
                           capture_output=True, text=True)
        print("telemetry push rc:", q.returncode, flush=True)
    except Exception as e:
        print("telemetry push failed:", str(e)[:200], flush=True)
push_telemetry()
assert p.returncode == 0, f"KERNEL CRASHED (exit {p.returncode}) — log tail is above; telemetry has it"

In [ ]:
import json, math
r = json.load(open("/content/v40_report.json"))
if "alloc_ledger" not in r or not r.get("alloc_ledger"):
    print("PARTIAL REPORT (crash save) — no exchange verdict; see cell 2 exit code + telemetry")
else:
    best = r.get("best_val_ce"); ctrl = 2.2706
    led = r.get("alloc_ledger", []); ood = r.get("ood_probe", {}) or {}
    print("B1 market<=control:", best, "vs", ctrl, "->", "PASS" if (best is not None and best <= ctrl) else "FAIL")
    print("B2 OOD:", ood.get("n_distinct"), "distinct ->", "PASS" if (ood.get("ood_ok") and ood.get("n_distinct", 0) >= 18) else "FAIL")
    print("B3 floor:", "PASS" if led and min(min(l) for l in led) >= 0.049 else "FAIL", "| clearings:", len(led))
    print("B4 supply_resp:", r.get("supply_responsiveness"), "->", "PASS" if (r.get("supply_responsiveness") or 0) > 0.3 else "FAIL")
    print("B5 finite curve:", "PASS" if all(c.get("val_ce") is not None and math.isfinite(c["val_ce"]) for c in r.get("curve", [])) else "FAIL")
    print("alloc final:", r.get("mkt_final"))